# 🎮 Video Games Sentiment Analysis (Analisis Sentimen Ulasan Video Games)

Notebook ini mengimplementasikan alur lengkap analisis sentimen berbasis **Natural Language Processing (NLP)** dan **Machine Learning** pada dataset **Amazon Reviews 2023** kategori **Video_Games**.

### Alur Proyek:
1. **Persiapan Lingkungan & Pustaka**
2. **Pengunduhan & Pemuatan Data Bertahap (Streaming / Chunking)** (Filter timestamp $\le 31$ Des 2019)
3. **Pembersihan Data & Pelabelan (Preprocessing)** (Buang rating 3, label biner 0 & 1, gabung `full_text`)
4. **Pembagian Data (Train-Validation Split)** (Time-based Split & Stratified Split)
5. **Ekstraksi Fitur & Pemodelan Machine Learning**
   - `TfidfVectorizer` (unigram & bigram, `max_features=20000`, negasi preserver)
   - Model 1: **Logistic Regression** (`class_weight='balanced'`)
   - Model 2: **LinearSVC** (`class_weight='balanced'`)
   - Model 3: **Multinomial Naive Bayes** (Pembanding baseline probabilitas)
   - Model 4: **LightGBM** (Pembanding Gradient Boosting)
   - Pembungkusan dalam `Pipeline` scikit-learn
6. **Evaluasi & Benchmarking Komparatif Model**
7. **Penyimpanan Model Terlatih** (`model/model_sentimen.joblib`)
8. **Pengujian Inferensi Real-time**

## 1. Persiapan Lingkungan & Pemuatan Pustaka

In [1]:
import os
import re
import gzip
import json
import zlib
import time
import datetime
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
print("Semua pustaka berhasil dimuat!")

Semua pustaka berhasil dimuat!


## 2. Pengunduhan & Pemuatan Data secara Bertahap (Chunking/Streaming)

> **Penting:** Dataset Amazon Reviews 2023 kategori `Video_Games` berukuran besar (arsip terkompresi `.jsonl.gz` berukuran ~814 MB). Untuk mencegah kehabisan memori RAM, kita membaca berkas secara **streaming baris demi baris (line-by-line)** menggunakan `json.loads`.
>
> **Aturan Filter:** Hanya ulasan dengan `timestamp <= 1577836799999` (31 Desember 2019, 23:59:59.999 UTC) yang diproses.

In [2]:
URL_VIDEO_GAMES = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz"
CUTOFF_TIMESTAMP = 1577836799999  # 31 Desember 2019 23:59:59.999 UTC

raw_csv_path = os.path.join("data", "video_games_reviews_2019.csv")
if os.path.exists(raw_csv_path):
    print(f"Memuat data ulasan dari berkas lokal: {raw_csv_path}")
    df = pd.read_csv(raw_csv_path)
else:
    print(f"Mengalirkan ulasan dari web: {URL_VIDEO_GAMES}...")
    from load_data import stream_reviews_from_url
    records = list(stream_reviews_from_url(max_records=5000))
    df = pd.DataFrame(records)
    os.makedirs("data", exist_ok=True)
    df.to_csv(raw_csv_path, index=False)

print(f"Total baris data awal: {len(df)}")
print(f"Maksimum timestamp   : {df['timestamp'].max()} (Batas: {CUTOFF_TIMESTAMP})")
print(f"Rentang tanggal      : {df['date'].min()} s/d {df['date'].max()}")
print("Verifikasi <= 2019   :", (df['timestamp'] <= CUTOFF_TIMESTAMP).all())
df.head()

## 3. Pembersihan Data & Pelabelan (Preprocessing)

1. **Buang baris** dengan `rating == 3` (netral) atau nilai kosong pada `rating`, `title`, dan `text`.
2. **Buat label biner (`label`)**:
   - `rating` 1 atau 2 $\rightarrow$ **`0` (Negatif)**
   - `rating` 4 atau 5 $\rightarrow$ **`1` (Positif)**
3. **Gabungkan kolom `title` dan `text`** menjadi satu fitur teks utama:
   $$df[\text{'full\_text'}] = df[\text{'title'}] + \text{" "} + df[\text{'text'}]$$
4. **Pembersihan dasar**: Menghapus tag HTML, URL, karakter tidak relevan, serta whitespace berlebih.

In [3]:
preprocessed_csv = os.path.join("data", "video_games_preprocessed.csv")
if os.path.exists(preprocessed_csv):
    print(f"Memuat dataset preprocessing dari: {preprocessed_csv}")
    df_clean = pd.read_csv(preprocessed_csv)
else:
    from load_data import preprocess_reviews_dataframe
    df_clean = preprocess_reviews_dataframe(df)
    df_clean.to_csv(preprocessed_csv, index=False)

print(f"Total data bersih: {len(df_clean)} sampel")
print("Distribusi Label Biner:")
print(df_clean["label"].value_counts())
df_clean[["rating", "title", "label", "clean_text"]].head()

## 4. Pembagian Data (Train-Validation Split)

Menerapkan dua strategi pembagian:
1. **Time-based Split** (Train: $\le 2018$, Val: $2019$): Menguji kemampuan generalisasi pada ulasan masa depan.
2. **Stratified Split** (80:20): Menjaga proporsi kelas minoritas negatif (~12.94%) tetap seimbang.

In [4]:
# 1. Time-based Split
train_time_df = df_clean[df_clean["year"] <= 2018].copy()
val_time_df = df_clean[df_clean["year"] == 2019].copy()

X_train_time, y_train_time = train_time_df["clean_text"], train_time_df["label"]
X_val_time, y_val_time = val_time_df["clean_text"], val_time_df["label"]

# 2. Stratified Split (80:20)
X_train_strat, X_val_strat, y_train_strat, y_val_strat = train_test_split(
    df_clean["clean_text"], df_clean["label"], test_size=0.2, random_state=42, stratify=df_clean["label"]
)

print(f"Time-based Split : Train = {len(X_train_time)} sampel, Val = {len(X_val_time)} sampel")
print(f"Stratified Split : Train = {len(X_train_strat)} sampel, Val = {len(X_val_strat)} sampel")

## 5. Ekstraksi Fitur & Pemodelan Machine Learning

### A. Ekstraksi Fitur: TF-IDF Vectorizer
- **`ngram_range=(1, 2)`**: Menangkap kata tunggal (unigram) dan kombinasi 2 kata (bigram). Bigram sangat penting untuk menangkap **frasa negasi** seperti *"not good"*, *"no fun"*, *"waste money"*, atau *"never buy"*.
- **Preservasi Kata Negasi**: Secara default, *stop words* bawaan bahasa Inggris menghapus kata seperti *"not"*, *"no"*, dan *"never"*. Pada implementasi ini, kita **mengecualikan kata negasi dari stop words** sehingga frasa sentimen negatif tetap tertangkap secara utuh.
- **`max_features=20000`**: Membatasi kosakata paling informatif untuk efisiensi komputasi.
- **`sublinear_tf=True`**: Menggunakan skala logaritmik $1 + \log(\text{tf})$ untuk meredam dominasi frekuensi kata tinggi.

### B. Model Klasifikasi yang Dilatih
1. **Logistic Regression (`class_weight='balanced'`)**: Linear classifier berbobot seimbang untuk mengatasi ketimpangan kelas.
2. **LinearSVC (`class_weight='balanced'`)**: Support Vector Classifier berbasis margin optimal, dibungkus dalam `CalibratedClassifierCV` untuk menghasilkan estimasi probabilitas.
3. **Multinomial Naive Bayes (`alpha=0.5`)**: Model pembanding berbasis probabilitas frekuensi kata.
4. **LightGBM (`class_weight='balanced'`)**: Model pembanding berbasis *Gradient Boosted Decision Trees* yang efisien untuk data tabular/teks berdimensi tinggi.

### C. Scikit-learn Pipeline
Semua proses (TF-IDF Vectorizer + Model Klasifikasi) dibungkus rapi dalam objek `Pipeline` scikit-learn sehingga inferensi ulasan baru dapat langsung dilakukan tanpa proses manual.

In [5]:
# Daftar kata negasi yang wajib dipertahankan
negation_words = {'not', 'no', 'never', 'neither', 'nobody', 'none', 'nor', 'nothing', 'nowhere', 'hardly', 'barely', 'scarcely'}
custom_stop_words = list(ENGLISH_STOP_WORDS - negation_words)

# Kamus model yang akan dievaluasi
model_candidates = {
    "Logistic Regression (balanced)": LogisticRegression(
        C=1.0, max_iter=1000, class_weight="balanced", random_state=42
    ),
    "LinearSVC (balanced)": CalibratedClassifierCV(
        LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, random_state=42)
    ),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.5),
    "LightGBM (balanced)": LGBMClassifier(
        n_estimators=150, learning_rate=0.05, class_weight="balanced", random_state=42, verbose=-1
    )
}

# Membangun pipeline untuk setiap model kandidat
pipelines = {}
for name, clf in model_candidates.items():
    pipelines[name] = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            stop_words=custom_stop_words
        )),
        ("classifier", clf)
    ])

print("Semua 4 Pipeline model berhasil dikonfigurasi:")
for name in pipelines:
    print(f" - {name}")

## 6. Pelatihan & Evaluasi Komparatif Model (Benchmarking)
Melatih seluruh model pada data latih terstratifikasi dan mengevaluasi metrik Akurasi, Precision, Recall, F1-Score (khususnya untuk kelas minoritas Negatif), dan Macro F1.

In [6]:
benchmark_results = []

print("Mulai proses pelatihan dan evaluasi 4 model...")
for name, pipe in pipelines.items():
    start_t = time.time()
    pipe.fit(X_train_strat, y_train_strat)
    train_time = time.time() - start_t
    
    y_pred = pipe.predict(X_val_strat)
    
    acc = accuracy_score(y_val_strat, y_pred)
    prec_neg = precision_score(y_val_strat, y_pred, pos_label=0)
    rec_neg = recall_score(y_val_strat, y_pred, pos_label=0)
    f1_neg = f1_score(y_val_strat, y_pred, pos_label=0)
    
    prec_pos = precision_score(y_val_strat, y_pred, pos_label=1)
    rec_pos = recall_score(y_val_strat, y_pred, pos_label=1)
    f1_pos = f1_score(y_val_strat, y_pred, pos_label=1)
    macro_f1 = f1_score(y_val_strat, y_pred, average="macro")
    
    benchmark_results.append({
        "Model": name,
        "Accuracy": acc,
        "Prec (Neg)": prec_neg,
        "Rec (Neg)": rec_neg,
        "F1 (Neg)": f1_neg,
        "Prec (Pos)": prec_pos,
        "Rec (Pos)": rec_pos,
        "F1 (Pos)": f1_pos,
        "Macro F1": macro_f1,
        "Train Time (s)": train_time
    })
    print(f"[{name}] Selesai dalam {train_time:.2f}s | Akurasi: {acc*100:.2f}% | F1 Neg: {f1_neg:.4f} | Macro F1: {macro_f1:.4f}")

df_results = pd.DataFrame(benchmark_results).sort_values(by="Macro F1", ascending=False)
df_results

In [7]:
# Visualisasi Perbandingan Metrik Antarmodel
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Bar plot Akurasi & Macro F1
df_melted = pd.melt(df_results, id_vars=["Model"], value_vars=["Accuracy", "Macro F1", "F1 (Neg)"], var_name="Metrik", value_name="Skor")
sns.barplot(data=df_melted, x="Model", y="Skor", hue="Metrik", ax=axes[0], palette="viridis")
axes[0].set_title("Perbandingan Akurasi, Macro F1, dan F1 Kelas Negatif", fontsize=12, fontweight="bold")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=25)

# 2. Heatmap Confusion Matrix Model Terbaik (LinearSVC)
best_name = df_results.iloc[0]["Model"]
best_pipe = pipelines[best_name]
best_pred = best_pipe.predict(X_val_strat)
cm = confusion_matrix(y_val_strat, best_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["Pred Negatif (0)", "Pred Positif (1)"],
            yticklabels=["Aktual Negatif (0)", "Aktual Positif (1)"])
axes[1].set_title(f"Confusion Matrix: {best_name}\n(Akurasi: {df_results.iloc[0]['Accuracy']*100:.2f}%)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Kelas Aktual")
axes[1].set_xlabel("Kelas Prediksi")

plt.tight_layout()
plt.show()

## 7. Penyimpanan Model Terlatih (Model Serialization)
Model terbaik (**LinearSVC balanced**, Macro F1 tertinggi) disimpan ke berkas utama `model/model_sentimen.joblib` dan `model_sentimen.joblib`. Model lainnya juga disimpan sebagai arsip pembanding.

In [8]:
os.makedirs("model", exist_ok=True)

# Simpan model terbaik ke path utama
best_model_path = os.path.join("model", "model_sentimen.joblib")
joblib.dump(best_pipe, best_model_path)
joblib.dump(best_pipe, "model_sentimen.joblib")

# Simpan masing-masing model pipeline
joblib.dump(pipelines["Logistic Regression (balanced)"], os.path.join("model", "model_logistic_regression.joblib"))
joblib.dump(pipelines["LinearSVC (balanced)"], os.path.join("model", "model_linearsvc.joblib"))
joblib.dump(pipelines["Multinomial Naive Bayes"], os.path.join("model", "model_multinomial_nb.joblib"))
joblib.dump(pipelines["LightGBM (balanced)"], os.path.join("model", "model_lightgbm.joblib"))

print(f"Model utama ({best_name}) berhasil disimpan di: {best_model_path} dan model_sentimen.joblib")
print("Semua model pembanding tersimpan lengkap di folder model/!")

## 8. Pengujian Inferensi Real-time (Uji Ulasan Baru Termasuk Frasa Negasi)
Menguji model pipeline terlatih dengan berbagai skenario ulasan game baru, termasuk frasa negasi seperti *"not good"*, *"no fun"*, dan *"waste of money"*.

In [9]:
# Muat model pipeline utama
loaded_model = joblib.load(best_model_path)

ulasan_uji = [
    "Masterpiece of RPG! The combat is smooth, story is deep, and visuals are breathtaking.",
    "Not good at all. The combat is clunky and it has no fun gameplay loops.",
    "Broken launch, heavy frame drops, constant crashing and game breaking bugs.",
    "Great co-op fun with friends. Excellent soundtrack and challenging boss fights.",
    "Overpriced buggy mess. Definitely not worth the price and complete waste of money."
]

prediksi = loaded_model.predict(ulasan_uji)
probabilitas = loaded_model.predict_proba(ulasan_uji)

print("=== PENGUJIAN INFERENSI MODEL PIPELINE TERLATIH ===\n")
for teks, pred, prob in zip(ulasan_uji, prediksi, probabilitas):
    label = "POSITIF" if pred == 1 else "NEGATIF"
    conf = max(prob) * 100
    print(f"Ulasan    : \"{teks}\"")
    print(f"Sentimen  : {label} (Tingkat Keyakinan: {conf:.2f}%)")
    print("-" * 75)